# Exploratory analysis

Scratch space for inspecting matcher scores, confusion cases, and threshold tuning.

Run `python scripts/generate_data.py` first so `data/raw/` and
`data/processed/ground_truth.json` exist.

Everything here uses the **offline** stack, so it runs with no API key and no
model download.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from reconciler.data_loader import load_bank, load_invoices, load_settlements
from reconciler.engine import ReconciliationEngine
from reconciler.matchers.faiss_matcher import FaissMatcher, HashingEmbedder
from reconciler.matchers.llm_matcher import HeuristicLLMClient, LlmMatcher
from reconciler.matchers.rule_matcher import RuleMatcher
from reconciler.matchers.tfidf_matcher import TfidfMatcher
from reconciler.pipeline import ThreeWayReconciliationPipeline

raw = ROOT / "data" / "raw"
invoices = load_invoices(raw / "invoices.csv")
settlements = load_settlements(raw / "razorpay_settlements.csv")
bank = load_bank(raw / "bank_statement.csv")
len(invoices), len(settlements), len(bank)

## Run the pipeline

The engine is built explicitly rather than with defaults, so this notebook never
reaches for the network.

In [ ]:
engine = ReconciliationEngine(matchers=[
    RuleMatcher(),
    TfidfMatcher(),
    FaissMatcher(embedder=HashingEmbedder()),
    LlmMatcher(client=HeuristicLLMClient()),
])

result = ThreeWayReconciliationPipeline(engine=engine).run(invoices, settlements, bank)

print(f"complete triangles: {len(result.complete_triangles)}")
print(f"partial triangles:  {len(result.partial_triangles)}")
print(f"exceptions:         {len(result.exceptions)} worth {result.exception_value:,.2f}")

## Metrics against ground truth

In [ ]:
import json
import time

from reconciler.reporting.metrics import compute_metrics

ground_truth = json.loads((ROOT / "data" / "processed" / "ground_truth.json").read_text())

start = time.perf_counter()
result = ThreeWayReconciliationPipeline(engine=engine).run(invoices, settlements, bank)
elapsed = time.perf_counter() - start

metrics = compute_metrics(
    result, len(invoices), len(settlements), len(bank), elapsed, ground_truth=ground_truth
)
metrics

## Where each tier did its work

`tier_timings` reports wall-clock cost per tier, which is what shows whether the
escalation is actually paying off — the expensive tiers should be both rare and
cheap because they only ever see the residual.

In [ ]:
import pandas as pd

from reconciler.reporting.metrics import tier_timings

pd.DataFrame([t.__dict__ for t in tier_timings(result)]).set_index("tier")

## Inspect the exception queue

Ranked by value at risk. `best_candidate_id` is the nearest record the engine
considered for an unmatched record, or the leg that *did* resolve for a partial
triangle.

In [ ]:
from reconciler.reporting.exceptions_report import exceptions_to_dataframe, summarize

print(summarize(result.exceptions))
exceptions_to_dataframe(result.exceptions).head(15)

## Threshold tuning

Sweep the TF-IDF tier's `confidence_threshold` and watch where the work goes.

The interesting column is not `match_rate` — it is the tier breakdown. As the
threshold rises, TF-IDF declines more pairs and they **escalate to FAISS** rather
than being lost, while precision and recall stay at 1.0. That cascade absorbing a
stricter upstream bar is the whole design working; if tightening a tier dropped
the achievable match rate instead, the tier below it would not be doing its job.

A change that raises match rate while dropping precision is making the engine
confidently wrong, which is worse than leaving the record in the review queue.

In [ ]:
rows = []
for threshold in [0.45, 0.70, 0.85, 0.90, 0.95, 0.99]:
    tfidf = TfidfMatcher()
    tfidf.confidence_threshold = threshold
    tuned = ReconciliationEngine(matchers=[
        RuleMatcher(),
        tfidf,
        FaissMatcher(embedder=HashingEmbedder()),
        LlmMatcher(client=HeuristicLLMClient()),
    ])
    swept = ThreeWayReconciliationPipeline(engine=tuned).run(invoices, settlements, bank)
    m = compute_metrics(
        swept, len(invoices), len(settlements), len(bank), 1.0, ground_truth=ground_truth
    )
    rows.append({
        "threshold": threshold,
        "match_rate": m.match_rate,
        "achievable": m.achievable_match_rate,
        "precision": m.accuracy.precision,
        "recall": m.accuracy.recall,
        "tfidf": m.tier_breakdown["tfidf"],
        "faiss": m.tier_breakdown["faiss"],
        "llm": m.tier_breakdown["llm"],
    })

pd.DataFrame(rows).set_index("threshold")